# NB2 — Data Cleaning Pipeline
This notebook demonstrates a practical cleaning workflow on the Titanic dataset.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler

sns.set_theme(style='whitegrid')
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
df = pd.read_csv(url).copy()
df.head()


## Create a deliberately dirtier copy


In [ ]:
dirty = pd.concat([df, df.iloc[[0, 1]]], ignore_index=True)
dirty.loc[5, 'Sex'] = 'Male'
dirty.loc[10, 'Embarked'] = None
dirty.loc[15, 'Fare'] = dirty['Fare'].max() * 5
print('Shape before cleaning:', dirty.shape)
dirty.head()


## Duplicate handling


In [ ]:
print('Duplicate rows:', dirty.duplicated().sum())
dirty = dirty.drop_duplicates().copy()
print('Shape after duplicate removal:', dirty.shape)


## Missing-value analysis and simple imputations


In [ ]:
dirty.isna().sum().sort_values(ascending=False)


In [ ]:
dirty['Sex'] = dirty['Sex'].astype(str).str.strip().str.lower()
dirty['Age'] = dirty['Age'].fillna(dirty['Age'].median())
dirty['Embarked'] = dirty['Embarked'].fillna(dirty['Embarked'].mode()[0])
dirty['CabinMissing'] = dirty['Cabin'].isna().astype(int)
dirty = dirty.drop(columns=['Cabin'])
dirty.head()


## Outlier detection with IQR


In [ ]:
Q1 = dirty['Fare'].quantile(0.25)
Q3 = dirty['Fare'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
outliers = dirty[(dirty['Fare'] < lower) | (dirty['Fare'] > upper)]
print('Fare outliers:', len(outliers))
dirty['Fare_capped'] = dirty['Fare'].clip(lower=lower, upper=upper)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(x=dirty['Fare'], ax=axes[0])
axes[0].set_title('Original Fare')
sns.boxplot(x=dirty['Fare_capped'], ax=axes[1])
axes[1].set_title('Capped Fare')
plt.tight_layout()
plt.show()


## Encoding and scaling


In [ ]:
clean = pd.get_dummies(dirty, columns=['Sex', 'Embarked', 'Pclass'], drop_first=True)
scale_cols = ['Age', 'Fare_capped', 'SibSp', 'Parch']

minmax = MinMaxScaler()
standard = StandardScaler()

scaled_minmax = pd.DataFrame(minmax.fit_transform(clean[scale_cols]), columns=[f'{c}_minmax' for c in scale_cols])
scaled_standard = pd.DataFrame(standard.fit_transform(clean[scale_cols]), columns=[f'{c}_zscore' for c in scale_cols])
pd.concat([scaled_minmax.head(), scaled_standard.head()], axis=1)


## Reusable pipeline function


In [ ]:
def full_cleaning_pipeline(input_df):
    data = input_df.copy()
    data = data.drop_duplicates()
    data['Sex'] = data['Sex'].astype(str).str.strip().str.lower()
    data['Age'] = data['Age'].fillna(data['Age'].median())
    data['Embarked'] = data['Embarked'].fillna(data['Embarked'].mode()[0])
    data['CabinMissing'] = data['Cabin'].isna().astype(int)
    data = data.drop(columns=['Cabin'])
    q1 = data['Fare'].quantile(0.25)
    q3 = data['Fare'].quantile(0.75)
    iqr = q3 - q1
    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr
    data['Fare_capped'] = data['Fare'].clip(lower=lower_fence, upper=upper_fence)
    data = pd.get_dummies(data, columns=['Sex', 'Embarked', 'Pclass'], drop_first=True)
    return data

pipeline_result = full_cleaning_pipeline(df)
pipeline_result.head()


## Practice prompts
1. When is deleting outliers a bad idea?
2. Why should scaling happen after train-test split in a real ML pipeline?
3. Why is one-hot encoding safer than label encoding for nominal variables?
